<a href="https://colab.research.google.com/github/R0dr1g0-23/Algoritmo-e-Estrutura-de-Dados/blob/main/Union_FInd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conjuntos Disjuntos: Union-Find — Tutorial

**Disciplina:** Algoritmos e Estruturas de Dados  
**DCOMP — Universidade Federal de Sergipe**  
**Referência:** CLRS, Capítulo 19

## Objetivos

Ao final deste tutorial você será capaz de:

- Implementar Union-Find com array de inteiros em Java
- Aplicar Union by Rank para controlar a altura da árvore
- Aplicar Path Compression para achatar o caminho durante Find
- Criar uma implementação genérica `UnionFind<T>` com `HashMap`
- Utilizar Union-Find para resolver problemas de conectividade (Kruskal, percolação)

> **Kernel:** [IJava](https://github.com/SpencerPark/IJava) — Java 17+

---
## 1. Estrutura básica: floresta de conjuntos disjuntos

Cada conjunto é representado como uma árvore enraizada.
Cada nó guarda apenas o ponteiro para o **pai**.
A **raiz** aponta para si mesma e é o representante do conjunto.

Operações:
- `makeSet(x)`: cria conjunto `{x}`, custo O(1)
- `find(x)`: retorna a raiz da árvore de `x`, custo O(altura)
- `union(x, y)`: conecta as raízes de `x` e `y`, custo O(1) + 2×find

In [1]:
class BasicUnionFind {
    int[] parent;

    BasicUnionFind(int n) {
        parent = new int[n];
        for (int i = 0; i < n; i++) parent[i] = i; // Make-Set
    }

    int find(int x) {
        // Segue ponteiros até raiz - sem compressão
        while (parent[x] != x) x = parent[x];
        return x;
    }

    void union(int x, int y) {
        int rx = find(x), ry = find(y);
        if (rx != ry) parent[ry] = rx; // sem rank: qualquer escolha
    }

    boolean connected(int x, int y) { return find(x) == find(y); }
}

// Teste básico
var uf = new BasicUnionFind(6);
System.out.println("Antes: connected(0,1)=" + uf.connected(0,1));
uf.union(0, 1);
uf.union(1, 2);
System.out.println("Após Union(0,1) e Union(1,2):");
System.out.println("  connected(0,2)=" + uf.connected(0,2)); // transitivo!
System.out.println("  connected(0,3)=" + uf.connected(0,3)); // ainda não

SyntaxError: invalid syntax (3127945905.py, line 1)

In [2]:
// Exercício 1: Verificar componentes conexas
// TODO: dado um grafo com `n` vértices e lista de arestas,
// retornar o número de componentes conexas.


    static int countComponents(int n, int[][] edges) {
        // TODO: use BasicUnionFind
        // Dica: inicializar com n componentes;
        //       cada union bem-sucedido (rx != ry) reduz em 1.
        BasicUnionFind vetor = new BasicUnionFind(n);
        int componentes = n;

        for(int i = 0; i < edges.length; i++){
            int x = edges[i][0];
            int y = edges[i][1];
            int rx = vetor.find(x);
            int ry = vetor.find(y);

            if(rx != ry){
                vetor.union(x,y);
                componentes--;
            }
        }

        return componentes; // substitua
}

// Testes automáticos
assert countComponents(5, new int[][]{{0,1},{1,2},{3,4}}) == 2
    : "Esperado 2 componentes";
assert countComponents(4, new int[][]{{0,1},{1,2},{2,3}}) == 1
    : "Esperado 1 componente";
assert countComponents(3, new int[][]{}) == 3
    : "Esperado 3 componentes (sem arestas)";
System.out.println("Exercício 1: OK!");

SyntaxError: invalid syntax (532989915.py, line 1)

---
## 2. Heurística 1: Union by Rank

O **rank** de um nó é um limite superior da altura da sua subárvore.

**Regra de union by rank:**
- Se `rank[rx] > rank[ry]`: `ry.parent = rx` (rank não muda)
- Se `rank[rx] < rank[ry]`: `rx.parent = ry` (rank não muda)
- Se iguais: qualquer escolha, mas incrementar rank do novo pai

**Resultado:** árvore de rank `k` tem pelo menos `2^k` nós → altura ≤ ⌊lg n⌋

In [ ]:
// Union by Rank — sem path compression
class RankUnionFind {
    int[] parent, rank;

    RankUnionFind(int n) {
        parent = new int[n]; rank = new int[n];
        for (int i = 0; i < n; i++) parent[i] = i;
    }

    int find(int x) {
        while (parent[x] != x) x = parent[x];
        return x;
    }

    boolean union(int x, int y) {
        int rx = find(x), ry = find(y);
        if (rx == ry) return false;
        // Garantir que rx tenha rank >= ry
        if (rank[rx] < rank[ry]) { int t = rx; rx = ry; ry = t; }
        parent[ry] = rx;
        if (rank[rx] == rank[ry]) rank[rx]++;
        return true;
    }

    int height(int root) {
        // Calcula altura real da subárvore de root
        int max = 0;
        for (int i = 0; i < parent.length; i++) {
            if (find(i) == root && i != root) {
                int d = 0; var cur = i;
                while (parent[cur] != cur) { d++; cur = parent[cur]; }
                max = Math.max(max, d);
            }
        }
        return max;
    }
}

// Demonstração: 8 elementos, unions sequenciais
var ruf = new RankUnionFind(8);
ruf.union(0,1); ruf.union(2,3); ruf.union(4,5); ruf.union(6,7);
ruf.union(0,2); ruf.union(4,6);
ruf.union(0,4);

int root = ruf.find(0);
System.out.println("Raiz: " + root);
System.out.println("Rank da raiz: " + ruf.rank[root]);
System.out.println("Altura real: " + ruf.height(root));
System.out.println("8 nós → altura ≤ lg(8) = 3: " + (ruf.height(root) <= 3));

---
## 3. Heurística 2: Path Compression

Durante `find(x)`, todos os nós no caminho de `x` até a raiz passam a apontar **diretamente** para a raiz.

Isso achata a árvore progressivamente, tornando futuras consultas mais baratas.

**Versão recursiva** (elegante):
```java
int find(int x) {
    if (parent[x] != x) parent[x] = find(parent[x]);
    return parent[x];
}
```

**Versão iterativa dois percursos** (preferível para pilhas grandes).

In [ ]:
// Path Compression — demonstração visual
class PCUnionFind {
    int[] parent, rank;

    PCUnionFind(int n) {
        parent = new int[n]; rank = new int[n];
        for (int i = 0; i < n; i++) parent[i] = i;
    }

    // Find com path compression recursivo
    int find(int x) {
        if (parent[x] != x)
            parent[x] = find(parent[x]); // comprime no retorno
        return parent[x];
    }

    void unionRaw(int rx, int ry) { parent[ry] = rx; } // union sem rank (demo)

    String pathTo(int x) {
        var sb = new StringBuilder();
        while (parent[x] != x) { sb.append(x).append("→"); x = parent[x]; }
        sb.append(x); return sb.toString();
    }
}

// Criar cadeia: 5→4→3→2→1→0 (0 é raiz)
var pcuf = new PCUnionFind(6);
for (int i = 1; i <= 5; i++) pcuf.unionRaw(i-1, i); // i aponta para i-1

System.out.println("Antes de find(5):");
for (int i = 0; i <= 5; i++)
    System.out.println("  caminho de " + i + ": " + pcuf.pathTo(i));

System.out.println("\nExecutando find(5)...");
int root = pcuf.find(5);
System.out.println("Raiz: " + root);

System.out.println("\nApós find(5) — path compression:");
for (int i = 0; i <= 5; i++)
    System.out.println("  parent[" + i + "] = " + pcuf.parent[i]);

In [ ]:
// Exercício 3: Path Compression iterativo (dois percursos)
// TODO: implemente find_iterativo que faz o mesmo que find recursivo
// mas sem recursão (dois percursos: 1) encontrar raiz, 2) comprimir)

class PCIterUnionFind extends PCUnionFind {
    PCIterUnionFind(int n) { super(n); }

    @Override
    int find(int x) {
    // Percurso 1: encontrar a raiz
    int root = x;
    while (parent[root] != root) {
        root = parent[root];
    }

    // Percurso 2: comprimir o caminho, fazendo todos apontarem pra raiz
    while (parent[x] != root) {
        int proximo = parent[x]; // guarda o próximo antes de sobrescrever
        parent[x] = root;
        x = proximo;
    }

    return root;
    }
}

// Teste automático
var pciter = new PCIterUnionFind(6);
for (int i = 1; i <= 5; i++) pciter.unionRaw(i-1, i);
int r = pciter.find(5);
Após find(5), todos devem apontar para 0
assert pciter.parent[5] == 0 : "5 deve apontar para 0";
assert pciter.parent[4] == 0 : "4 deve apontar para 0";
assert pciter.parent[3] == 0 : "3 deve apontar para 0";
System.out.println("Exercício 3: implemente find iterativo e descomente os asserts.");

---
## 4. Implementação completa: Rank + Path Compression

Combinando as duas heurísticas:
- **Union by Rank**: controla a altura de longo prazo
- **Path Compression**: achata o caminho em cada Find

Resultado: $O(m\,\alpha(n))$ amortizado — praticamente $O(1)$ por operação.

In [ ]:
// Implementação completa: Rank + Path Compression
class UnionFind {
    private final int[] parent;
    private final int[] rank;
    private int components;

    public UnionFind(int n) {
        parent = new int[n]; rank = new int[n];
        for (int i = 0; i < n; i++) parent[i] = i;
        components = n;
    }

    // Find com path compression recursivo
    public int find(int x) {
        if (parent[x] != x) parent[x] = find(parent[x]);
        return parent[x];
    }

    // Union by rank
    public boolean union(int x, int y) {
        int rx = find(x), ry = find(y);
        if (rx == ry) return false;
        if (rank[rx] < rank[ry]) { int t = rx; rx = ry; ry = t; }
        parent[ry] = rx;
        if (rank[rx] == rank[ry]) rank[rx]++;
        components--;
        return true;
    }

    public boolean connected(int x, int y) { return find(x) == find(y); }
    public int components() { return components; }
}

// Demo: 10 elementos, sequência de unions
var uf = new UnionFind(10);
System.out.println("Componentes iniciais: " + uf.components()); // 10

uf.union(0,1); uf.union(2,3); uf.union(4,5);
System.out.println("Após 3 unions: " + uf.components()); // 7

uf.union(0,2); uf.union(4,6); uf.union(0,4);
System.out.println("Após +3 unions: " + uf.components()); // 4

System.out.println("connected(1,3): " + uf.connected(1,3)); // true
System.out.println("connected(1,7): " + uf.connected(1,7)); // false

In [ ]:
// Exercício 4: GenericUnionFind<T> com HashMap
// TODO: implemente uma versão genérica que aceita qualquer tipo T

import java.util.*;

class GenericUnionFind<T> {
    private final Map<T, T>       parent = new HashMap<>();
    private final Map<T, Integer> rank   = new HashMap<>();

    public void makeSet(T x) {
        // TODO
        parent.put(x, x);
        rank.put(x, 0);
    }

    public T find(T x) {
        T p = parent.get(x);
        if (!p.equals(x)) {
            T raiz = find(p);
            parent.put(x, raiz);
            return raiz;
        }
        return x;
    }

    public boolean union(T x, T y) {
        // TODO: com union by rank
        T rx = find(x), ry = find(y);
        if (rx.equals(ry)) return false;

        int rankX = rank.get(rx);
        int rankY = rank.get(ry);

        if (rankX < rankY) { T t = rx; rx = ry; ry = t; }

        parent.put(ry, rx);
        if (rankX == rankY) {
            rank.put(rx, rank.get(rx) + 1);
        }
        return true
    }

    public boolean connected(T x, T y) { return find(x).equals(find(y)); }
}

// Teste com Strings
var guf = new GenericUnionFind<String>();
List.of("A","B","C","D","E").forEach(guf::makeSet);
guf.union("A","B"); guf.union("B","C");
assert guf.connected("A","C") : "A e C devem estar conectados";
assert !guf.connected("A","D") : "A e D nao devem estar conectados";
System.out.println("Exercício 4: implemente GenericUnionFind e descomente os asserts.");

---
## 5. Aplicação: Algoritmo de Kruskal

Union-Find é o componente central do algoritmo de Kruskal para Árvore Geradora Mínima (MST).

**Algoritmo:**
1. Ordenar todas as arestas por peso
2. Para cada aresta `(u, v, w)` em ordem crescente:
   - Se `find(u) ≠ find(v)`: adicionar à MST e `union(u, v)`
3. MST tem exatamente `n-1` arestas

In [ ]:
import java.util.*;

record Edge(int u, int v, double weight) implements Comparable<Edge> {
    @Override public int compareTo(Edge other) {
        return Double.compare(this.weight, other.weight);
    }
}

List<Edge> kruskal(int n, List<Edge> edges) {
    var uf  = new UnionFind(n);
    var mst = new ArrayList<Edge>();
    var sorted = new ArrayList<>(edges);
    Collections.sort(sorted);

    for (var e : sorted) {
        if (uf.union(e.u(), e.v()))  // false = ciclo
            mst.add(e);
        if (mst.size() == n - 1) break; // MST completa
    }
    return mst;
}

// Grafo exemplo: 5 vértices, 7 arestas
var edges = List.of(
    new Edge(0, 1, 2.0),
    new Edge(0, 3, 4.0),
    new Edge(1, 2, 3.0),
    new Edge(1, 3, 8.0),
    new Edge(2, 4, 6.0),
    new Edge(3, 4, 1.0),
    new Edge(1, 4, 7.0)
);

var mst = kruskal(5, edges);
System.out.println("MST ("+mst.size()+" arestas):");
double total = 0;
for (var e : mst) {
    System.out.printf("  %d -- %d  peso=%.1f%n", e.u(), e.v(), e.weight());
    total += e.weight();
}
System.out.printf("Peso total da MST: %.1f%n", total);

In [ ]:
// Exercício 5: Detectar ciclo
// TODO: implemente hasCycle usando UnionFind.
// Um ciclo existe quando union retorna false (vertices ja conectados).

boolean hasCycle(int n, int[][] edges) {
    // TODO
    UnionFind uf = new UnionFind(n);

    for (int i = 0; i < edges.length; i++) {
        int x = edges[i][0];
        int y = edges[i][1];
        if (!uf.union(x, y)) {
            return true;
        }
    }

    return false; // passou por todas as arestas sem repetir conexão
}

//Testes
assert hasCycle(3, new int[][]{{0,1},{1,2},{2,0}}) : "Triangulo tem ciclo";
assert !hasCycle(3, new int[][]{{0,1},{1,2}}) : "Caminho nao tem ciclo";
assert !hasCycle(4, new int[][]{{0,1},{2,3},{1,3}}) : "Arvore nao tem ciclo";
System.out.println("Exercício 5: implemente hasCycle e descomente os asserts.");

---
## 6. Aplicação: Percolação

Um sistema de **percolação** modela se fluido pode fluir do topo até a base de uma grade.

- Grade `n × n`; cada célula é aberta com probabilidade `p`
- Sistema percola ↔ existe caminho de células abertas de linha 0 até linha `n-1`
- Threshold crítico: `p* ≈ 0.593` (estimado por Monte Carlo)

**Modelagem com Union-Find:**
- Nós `0..n²-1` para células
- Nó virtual `top = n²` conectado a todas as células abertas da linha 0
- Nó virtual `bot = n²+1` conectado a todas as células abertas da linha `n-1`
- Percola ↔ `connected(top, bot)`

In [ ]:
class Percolation {
    private final int n;
    private final boolean[] open;
    private final UnionFind uf;
    private final int TOP, BOT;

    Percolation(int n) {
        this.n = n;
        open = new boolean[n * n];
        TOP = n * n; BOT = n * n + 1;
        uf = new UnionFind(n * n + 2);
    }

    int idx(int row, int col) { return row * n + col; }

    void openCell(int row, int col) {
        int id = idx(row, col);
        if (open[id]) return;
        open[id] = true;

        if (row == 0)   uf.union(id, TOP);
        if (row == n-1) uf.union(id, BOT);

        int[][] dirs = {{-1,0},{1,0},{0,-1},{0,1}};
        for (var d : dirs) {
            int r2 = row + d[0], c2 = col + d[1];
            if (r2 >= 0 && r2 < n && c2 >= 0 && c2 < n && open[idx(r2, c2)])
                uf.union(id, idx(r2, c2));
        }
    }

    boolean percolates() { return uf.connected(TOP, BOT); }
}

// Simulação Monte Carlo: estimar threshold de percolação
import java.util.Random;

int trials = 1000, gridSize = 20;
double sumP = 0;
var rng = new Random(42);

for (int t = 0; t < trials; t++) {
    var perc = new Percolation(gridSize);
    var cells = new ArrayList<int[]>();
    for (int i = 0; i < gridSize; i++)
        for (int j = 0; j < gridSize; j++)
            cells.add(new int[]{i, j});
    Collections.shuffle(cells, rng);

    int opened = 0;
    for (var cell : cells) {
        perc.openCell(cell[0], cell[1]);
        opened++;
        if (perc.percolates()) break;
    }
    sumP += (double) opened / (gridSize * gridSize);
}

System.out.printf("Threshold estimado (grade %dx%d, %d trials): %.4f%n",
    gridSize, gridSize, trials, sumP / trials);
System.out.println("Valor teórico: ≈ 0.5927");

---
## Desafio Final: Union-Find com histórico de operações

Implemente uma classe `PersistentUnionFind` que:

1. Suporta todas as operações de `UnionFind`
2. Permite **desfazer** a última operação `union` (`rollback()`)
3. A história deve ser armazenada como uma pilha de operações de union

**Dica:** Para suportar rollback, **não** use path compression (ela modifica muitos ponteiros). Use apenas Union by Rank e guarde na pilha `(ry, rx_antigo_rank)` para desfazer.

Complexidade esperada: Find $O(\log n)$, Union e Rollback $O(\log n)$.

In [3]:
import java.util.*;

class PersistentUnionFind {
    private final int[] parent;
    private final int[] rank;
    // Pilha: cada elemento é {ry, rx_original_rank, rx}
    private final Deque<int[]> history = new ArrayDeque<>();

    PersistentUnionFind(int n) {
        parent = new int[n]; rank = new int[n];
        for (int i = 0; i < n; i++) parent[i] = i;
    }

    int find(int x) {
        // TODO: SEM path compression (para permitir rollback)
         while (parent[x] != x) {
            x = parent[x];
        }
        return x;
    }

    boolean union(int x, int y) {
        // TODO: union by rank + gravar operação na pilha
        int rx = find(x), ry = find(y);
        if (rx == ry) return false;

        if (rank[rx] < rank[ry]) { int t = rx; rx = ry; ry = t; }

        history.push(new int[]{ry, rx, rank[rx]});

        parent[ry] = rx;
        if (rank[rx] == rank[ry]) rank[rx]++;

        return true;
    }

    void rollback() {
        if (history.isEmpty()) return;

        int[] ultimo = history.pop();
        int ry = ultimo[0];
        int rx = ultimo[1];
        int rankAntigo = ultimo[2];

        parent[ry] = ry;
        rank[rx] = rankAntigo;
    }

    boolean connected(int x, int y) { return find(x) == find(y); }
}

// Testes
var puf = new PersistentUnionFind(5);
puf.union(0, 1);
puf.union(1, 2);
assert puf.connected(0, 2) : "0 e 2 devem estar conectados";
puf.rollback();
assert !puf.connected(0, 2) : "0 e 2 nao devem estar conectados apos rollback";
assert puf.connected(0, 1)  : "0 e 1 ainda devem estar conectados";
System.out.println("Desafio: implemente PersistentUnionFind e descomente os asserts.");

SyntaxError: invalid syntax (3122214598.py, line 1)

---
## Referências

Veja o arquivo `../referencias.bib` para a lista completa.

Principais:
- **CLRS 4ª ed.**, Capítulo 19 — *Data Structures for Disjoint Sets*
- **Tarjan (1975)** — artigo original da análise com $\alpha(n)$
- **Sedgewick & Wayne** — Algorithms, 4ª ed., Seção 1.5